In [0]:
%sql
CREATE CATALOG IF NOT EXISTS rcm_platform
COMMENT 'RCM Intelligence Platform — Healthcare Revenue Cycle Management. Bronze, Silver, Gold and ML assets.';

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# STEP 1 — CREATE ALL SCHEMAS UNDER rcm_platform
# ============================================================
create_schemas()

In [0]:
# ============================================================
# STEP 2 — CREATE PIPELINE AUDIT LOG TABLE
# ============================================================
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_AUDIT_PIPELINE_LOG} (
        run_id             STRING    NOT NULL COMMENT 'Unique ID for this pipeline run',
        pipeline_name      STRING    NOT NULL COMMENT 'Name of the pipeline',
        pipeline_version   STRING    NOT NULL COMMENT 'Version of the pipeline',
        environment        STRING    NOT NULL COMMENT 'dev | staging | prod',
        notebook_name      STRING    NOT NULL COMMENT 'Notebook that wrote this row',
        layer              STRING    NOT NULL COMMENT 'bronze | silver | gold | ml | setup',
        status             STRING    NOT NULL COMMENT 'SUCCESS | FAILED | WARNING',
        rows_read          INT                COMMENT 'Rows read from source',
        rows_written       INT                COMMENT 'Rows written to target',
        rows_quarantined   INT                COMMENT 'Rows sent to DQ quarantine',
        message            STRING             COMMENT 'Additional context or error detail',
        logged_at          TIMESTAMP NOT NULL COMMENT 'UTC timestamp of this log entry'
    )
    USING DELTA
    COMMENT 'Pipeline observability — one row per notebook execution'
""")

print(f"OK  {TBL_AUDIT_PIPELINE_LOG}")

In [0]:
# ============================================================
# STEP 3 — CREATE DQ QUARANTINE TABLE
# ============================================================
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_BRONZE_QUARANTINE} (
        _dq_fail_reason    STRING    COMMENT 'Pipe-delimited list of failed DQ checks',
        _source_table      STRING    COMMENT 'Table this row came from',
        _quarantined_at    TIMESTAMP COMMENT 'When this row was quarantined',
        _notebook          STRING    COMMENT 'Notebook that quarantined this row'
    )
    USING DELTA
    COMMENT 'DQ quarantine — failed rows stored here for investigation and reprocessing'
""")

print(f"OK  {TBL_BRONZE_QUARANTINE}")

In [0]:
# ============================================================
# STEP 4 — CREATE DBFS DIRECTORIES
# ============================================================
create_dbfs_dirs()

In [0]:
# ============================================================
# STEP 5 — VERIFICATION
# ============================================================
print("=" * 55)
print("  VERIFICATION — rcm_platform catalog structure")
print("=" * 55)

# Check schemas
print("\nSchemas:")
for schema in [SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD, SCHEMA_ML, SCHEMA_AUDIT]:
    result = spark.sql(f"SHOW SCHEMAS IN {CATALOG} LIKE '{schema}'").collect()
    status = "OK " if result else "MISSING"
    print(f"  {status}  {CATALOG}.{schema}")

# Check tables
print("\nTables:")
for table in [TBL_AUDIT_PIPELINE_LOG, TBL_BRONZE_QUARANTINE]:
    exists = table_exists(table)
    status = "OK " if exists else "MISSING"
    print(f"  {status}  {table}")

# Check Volume directories
print("\nVolume directories:")
for path in [BASE_VOLUME_PATH, RAW_PATH, CHECKPOINT_PATH]:
    try:
        dbutils.fs.ls(path)
        print(f"  OK   {path}")
    except Exception:
        print(f"  MISSING  {path}")

print("\n" + "=" * 55)

In [0]:
# ============================================================
# STEP 6 — LOG THIS SETUP RUN TO AUDIT TABLE
# ============================================================
log_pipeline_run(
    notebook_name    = "00_setup_databases",
    layer            = "setup",
    status           = "SUCCESS",
    rows_read        = 0,
    rows_written     = 0,
    rows_quarantined = 0,
    message          = "All schemas, tables and DBFS directories created successfully"
)